# Semana 30 · ¿Es la luz una onda electromagnética?

Este cuaderno acompaña a la sesión de proyecto de la semana 30: la velocidad de la luz a partir de constantes eléctricas, la forma de una onda electromagnética, el espectro electromagnético y la energía de los fotones.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. La velocidad de la luz a partir de la electricidad

$$c = \frac{1}{\sqrt{\mu_0\,\varepsilon_0}}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

eps0 = 8.8541878e-12     # permitividad del vacío, C²/(N·m²)
mu0 = 1.25663706e-6      # permeabilidad del vacío, N/A²
c = 1 / np.sqrt(mu0 * eps0)
print(f"c = {c:,.0f} m/s".replace(",", " "))

## 2. Una onda electromagnética en tres dimensiones

La onda avanza a lo largo del eje $x$. El campo eléctrico oscila en el eje $y$ (rojo) y el magnético en el eje $z$ (azul), en fase. Las flechas muestran el valor de cada campo a lo largo del camino en un instante.

In [ ]:
x = np.linspace(0, 4 * np.pi, 400)
E = np.sin(x)
B = np.sin(x)          # en unidades en las que E = cB y c = 1, para dibujarlos del mismo tamaño

fig = plt.figure(figsize=(11, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(x, E, 0 * x, "r", lw=2, label="campo eléctrico E")
ax.plot(x, 0 * x, B, "b", lw=2, label="campo magnético B")
for xi in x[::12]:
    ax.plot([xi, xi], [0, np.sin(xi)], [0, 0], "r", lw=0.6)
    ax.plot([xi, xi], [0, 0], [0, np.sin(xi)], "b", lw=0.6)
ax.plot(x, 0 * x, 0 * x, "k", lw=0.8)
ax.set_xlabel("dirección de avance"); ax.set_ylabel("E"); ax.set_zlabel("B")
ax.legend(); ax.view_init(elev=20, azim=-60)
plt.show()

## 3. El espectro electromagnético en escala logarítmica

Dibujamos las regiones del espectro en un eje logarítmico de longitud de onda y, debajo, una curva **esquemática** de cuánta radiación de cada longitud de onda llega desde el espacio hasta el suelo. La curva no es una medida real: solo marca dónde están las ventanas (el visible, parte del infrarrojo cercano y la radio) y dónde la atmósfera es opaca.

Para colorear la franja visible usamos una aproximación sencilla de los colores de cada longitud de onda.

In [ ]:
def color_aprox(lam):
    """Color RGB aproximado de una longitud de onda visible (nm). Aproximación clásica por tramos."""
    if 380 <= lam < 440: r, g, b = -(lam - 440) / 60, 0, 1
    elif lam < 490: r, g, b = 0, (lam - 440) / 50, 1
    elif lam < 510: r, g, b = 0, 1, -(lam - 510) / 20
    elif lam < 580: r, g, b = (lam - 510) / 70, 1, 0
    elif lam < 645: r, g, b = 1, -(lam - 645) / 65, 0
    elif lam <= 750: r, g, b = 1, 0, 0
    else: return (0, 0, 0)
    f = 0.3 + 0.7 * (lam - 380) / 40 if lam < 420 else (0.3 + 0.7 * (750 - lam) / 50 if lam > 700 else 1)
    return (r * f, g * f, b * f)

regiones = [("gamma", 1e-14, 1e-11), ("rayos X", 1e-11, 1e-8), ("ultravioleta", 1e-8, 3.8e-7),
            ("visible", 3.8e-7, 7.5e-7), ("infrarrojo", 7.5e-7, 1e-3), ("microondas", 1e-3, 1),
            ("radio", 1, 1e4)]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True, gridspec_kw={"height_ratios": [1, 1.4]})
colores = ["#5b2a86", "#3b4cc0", "#7b68ee", None, "#b22222", "#cd853f", "#556b2f"]
for (nombre, a, b), col in zip(regiones, colores):
    if col:
        a1.axvspan(a, b, color=col, alpha=0.6)
    a1.text(np.sqrt(a * b), 0.5, nombre, ha="center", va="center", fontsize=9, color="white" if col else "black", rotation=0 if nombre != "visible" else 90)
for lam in np.linspace(380, 750, 120):
    a1.axvspan(lam * 1e-9, (lam + 3.1) * 1e-9, color=color_aprox(lam))
a1.set_xscale("log"); a1.set_yticks([]); a1.set_xlim(1e-14, 1e4)
a1.set_title("El espectro electromagnético")

lam = np.logspace(-14, 4, 2000)
loglam = np.log10(lam)
llega = np.zeros_like(lam)
llega += np.exp(-((loglam + 6.25) / 0.17) ** 2)                 # ventana óptica
llega += 0.5 * np.exp(-((loglam + 5.5) / 0.25) ** 2)            # infrarrojo cercano, a trozos
llega += 0.95 / (1 + np.exp(-(loglam + 2) * 6)) / (1 + np.exp((loglam - 1.3) * 6))   # ventana de radio
a2.fill_between(lam, llega, color="gray", alpha=0.6)
a2.set_xscale("log"); a2.set_ylim(0, 1.1); a2.set_xlabel("longitud de onda (m)")
a2.set_ylabel("fracción que llega\nal suelo (esquema)")
plt.tight_layout(); plt.show()

## 4. La energía de los fotones

$$E = h f = \frac{h c}{\lambda}.$$

Calculamos la energía, en electronvoltios, de un fotón típico de cada región.

In [ ]:
h = 6.62607015e-34       # J·s
eV = 1.602176634e-19     # J
ejemplos = [("radio FM", 3.0), ("wifi / microondas", 0.122), ("infrarrojo térmico", 10e-6),
            ("rojo", 700e-9), ("verde", 530e-9), ("violeta", 400e-9),
            ("ultravioleta C", 250e-9), ("rayos X", 0.1e-9), ("rayos gamma", 1e-12)]
print(f"{'región':<22}{'λ (m)':>12}{'f (Hz)':>12}{'E (eV)':>14}")
for nombre, lam in ejemplos:
    f = c / lam
    print(f"{nombre:<22}{lam:>12.2e}{f:>12.2e}{h * f / eV:>14.3g}")

Romper un enlace químico típico requiere unos pocos eV. Por eso los fotones de ultravioleta y de rayos X pueden dañar moléculas, y los de radio, aunque haya muchísimos, no.

**Ejercicio.** Calcula cuántos fotones por segundo emite una emisora de radio de 10 kW a 100 MHz, y cuántos una bombilla LED que emite 1 W de luz verde.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
for nombre, P, lam in [("emisora de 10 kW a 100 MHz", 1e4, c / 1e8), ("1 W de luz verde", 1.0, 530e-9)]:
    print(f"{nombre}: {P / (h * c / lam):.2e} fotones por segundo")

## 5. Para ir más lejos: la ecuación de ondas en una cuerda

Resolvemos numéricamente $\partial^2 y/\partial t^2 = v^2\,\partial^2 y/\partial x^2$ con **diferencias finitas**: la segunda derivada se aproxima con tres puntos vecinos,

$$\frac{\partial^2 y}{\partial x^2} \approx \frac{y_{i+1} - 2y_i + y_{i-1}}{\Delta x^2},$$

y lo mismo en el tiempo. Partimos de un pulso en el centro de una cuerda con los extremos fijos. El pulso se divide en dos que viajan en sentidos opuestos, rebotan en los extremos (invertidos) y se cruzan sin destruirse.

In [ ]:
N, L, v = 400, 1.0, 1.0
x = np.linspace(0, L, N)
dx = x[1] - x[0]
dt = 0.5 * dx / v                     # paso de tiempo estable (condición de Courant)
r2 = (v * dt / dx) ** 2

y_ant = np.exp(-((x - 0.5) / 0.03) ** 2)      # pulso inicial, quieto
y = y_ant.copy()
fotos = {}
pasos = int(1.6 / dt)
for n in range(pasos):
    y_nuevo = np.zeros_like(y)
    y_nuevo[1:-1] = 2 * y[1:-1] - y_ant[1:-1] + r2 * (y[2:] - 2 * y[1:-1] + y[:-2])
    if n == 0:
        y_nuevo[1:-1] = y[1:-1] + 0.5 * r2 * (y[2:] - 2 * y[1:-1] + y[:-2])   # arranque con velocidad nula
    y_ant, y = y, y_nuevo
    t = (n + 1) * dt
    for t_foto in [0.1, 0.3, 0.5, 0.8, 1.0]:
        if abs(t - t_foto) < dt / 2:
            fotos[t_foto] = y.copy()

fig, axs = plt.subplots(len(fotos) + 1, 1, figsize=(9, 9), sharex=True)
axs[0].plot(x, np.exp(-((x - 0.5) / 0.03) ** 2)); axs[0].set_ylabel("t = 0")
for ax, (t_foto, foto) in zip(axs[1:], sorted(fotos.items())):
    ax.plot(x, foto); ax.set_ylabel(f"t = {t_foto}"); ax.set_ylim(-1.1, 1.1)
axs[-1].set_xlabel("posición en la cuerda")
plt.tight_layout(); plt.show()

## 6. Experimenta

1. En la tabla de fotones, añade la línea de 21 cm del hidrógeno y la luz de una farola de sodio (589 nm).
2. En la cuerda, empieza con dos pulsos de signo contrario y míralos cruzarse: en el instante en que coinciden, la cuerda queda plana. ¿Dónde está la energía en ese momento? (Pista: en la velocidad.)
3. Si tienes el resultado del experimento del horno de microondas, calcula $c$ y el error relativo respecto al valor de este cuaderno.

In [ ]:
# Tu experimento aquí